# Data Cleaning & Preprocessing Demo
### Sales Dataset

This notebook demonstrates cleaning and preprocessing of a real-world sales dataset.
Tasks include handling missing values, removing duplicates, fixing data types, and preparing data for analysis.

In [1]:
import pandas as pd 
import numpy as np

In [2]:
df = pd.read_csv('data/raw_sales_data.csv')

In [3]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [4]:
df.shape

(10000, 8)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [6]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


## Initial Observations
- Dataset contains missing values
- Some columns have incorrect data types
- Dataset may contain duplicate records
- Column names are not standardized


In [7]:
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [8]:
df.duplicated().sum()

0

In [9]:
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='object')

## Data Issues Identified
- Missing values in some columns
- Presence of duplicate records
- Column names are not standardized
- Some numeric/date columns stored as object type


In [10]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
df.columns

Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date'],
      dtype='object')

In [11]:
numeric_cols = ['quantity', 'price_per_unit', 'total_spent']
categorical_cols = ['transaction_id', 'item', 'payment_method', 'location']

In [12]:
for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace('[^0-9.]', '', regex=True)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col].fillna(df[col].median(), inplace=True)

In [13]:
df['quantity'] = df['quantity'].astype(int)

In [14]:
for col in categorical_cols:
    df[col] = df[col].astype(str).fillna('UNKNOWN')

In [15]:
df.isnull().sum()

transaction_id        0
item                  0
quantity              0
price_per_unit        0
total_spent           0
payment_method        0
location              0
transaction_date    159
dtype: int64

In [16]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  object        
 1   item              10000 non-null  object        
 2   quantity          10000 non-null  int32         
 3   price_per_unit    10000 non-null  float64       
 4   total_spent       10000 non-null  float64       
 5   payment_method    10000 non-null  object        
 6   location          10000 non-null  object        
 7   transaction_date  9540 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int32(1), object(4)
memory usage: 586.1+ KB


## Data Cleaning Summary
- Standardized column names
- Handled missing values using median and mode
- Removed duplicate records
- Fixed data types for date columns


In [19]:
# Detecting outliers using the IQR method
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

outliers = ((df[numeric_cols] < (Q1 - 1.5 * IQR)) | 
            (df[numeric_cols] > (Q3 + 1.5 * IQR))).sum()

outliers

quantity            0
price_per_unit      0
total_spent       259
dtype: int64

## Outlier Analysis
Outlier detection was performed using the IQR method on numeric columns.
No significant extreme values were detected, so no outlier treatment was applied.


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  object        
 1   item              10000 non-null  object        
 2   quantity          10000 non-null  int32         
 3   price_per_unit    10000 non-null  float64       
 4   total_spent       10000 non-null  float64       
 5   payment_method    10000 non-null  object        
 6   location          10000 non-null  object        
 7   transaction_date  9540 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int32(1), object(4)
memory usage: 586.1+ KB


In [21]:
df.describe()

,quantity,price_per_unit,total_spent,transaction_date
count,10000.000000,10000.00000,10000.00000,9540
mean,3.027100,2.95265,8.87795,2023-07-01 23:00:31.698113280
min,1.000000,1.00000,1.00000,2023-01-01 00:00:00
25%,2.000000,2.00000,4.00000,2023-04-01 00:00:00
50%,3.000000,3.00000,8.00000,2023-07-02 00:00:00
75%,4.000000,4.00000,12.00000,2023-10-02 00:00:00
max,5.000000,5.00000,25.00000,2023-12-31 00:00:00
std,1.384614,1.24396,5.86059,NaN


In [23]:
df.to_csv("cleaned_sales_data.csv", index=False)

## Final Output
The dataset has been fully cleaned and preprocessed.
It is now free from missing values, duplicates, and outliers,
and is ready for analysis or machine learning tasks.
